# Setup

In [ ]:
# Standard library imports
import os
import time
import warnings

# The Keras backend must be chosen BEFORE keras is imported
os.environ['KERAS_BACKEND'] = 'torch'

# Third-party imports: core data stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Third-party imports: modeling
import torch
import keras

# Third-party imports: the foundation models
import timesfm
from tirex import load_model as load_tirex
from chronos import BaseChronosPipeline

# Configuration & Settings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())

This section of code prepares the environment for time series forecasting using several Python libraries. It begins by importing necessary modules from both the standard library – such as `os`, `time`, and `warnings` – and various third-party packages.

The line `os.environ['KERAS_BACKEND'] = 'torch'` is crucial; it explicitly sets the Keras backend to PyTorch *before* Keras itself is imported. This ensures that Keras will utilize PyTorch for its underlying tensor operations, which impacts performance and compatibility. 

Following this, core data science libraries like `numpy` (for numerical computation), `pandas` (for data manipulation with DataFrames), and `matplotlib` (for plotting) are imported.  Then come the modeling tools: `torch` (PyTorch itself) and `keras`.

The code then imports specialized time series forecasting packages: `timesfm`, `tirex` (with a custom loading function named `load_tirex`), and `chronos` (specifically, the base pipeline class `BaseChronosPipeline`). These represent different frameworks or models for handling time series data. 

Configuration settings are applied next.  The code suppresses both `FutureWarning` messages and general warnings to keep the output clean during execution.

Finally, the script prints the installed versions of Keras and confirms which backend (PyTorch in this case) is currently being used by Keras. This provides verification that the environment has been set up as intended.

In [ ]:
# general settings
class CFG:
    data_folder = './data/'
    graph_folder = './graphs/'
    img_dim1 = 20
    img_dim2 = 10
    SEED = 42
    STORE = 1                          # the store whose items we forecast
    HERO_ITEM = 1                      # the single series used for close-ups
    ARENA_ITEMS = list(range(1, 11))   # ten series for the head-to-head
    HORIZON = 14                       # forecast two weeks ahead
    LOOKBACK = 84                      # window length for models we train ourselves
    CONTEXT = 512                      # history handed to the foundation models
    N_ORIGINS = 12                     # weekly walk-forward test origins
    QUANTILES = [0.1, 0.5, 0.9]        # quantile levels for probabilistic scoring

# display style
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (CFG.img_dim1, CFG.img_dim2)

np.random.seed(CFG.SEED)

# hardware: Apple GPU if present, plain CPU otherwise
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print("Computing on:", DEVICE)

This code defines configuration settings and prepares the environment for a time series forecasting project. It begins by creating a class named `CFG` to encapsulate various parameters, making them easily accessible throughout the script. 

Within `CFG`, several key variables are defined: `data_folder` specifies the directory where data is stored; `graph_folder` indicates where generated plots will be saved; `img_dim1` and `img_dim2` set the dimensions for figures; `SEED` establishes a random seed for reproducibility.  `STORE` identifies the specific store being analyzed, `HERO_ITEM` designates a particular item for detailed examination, and `ARENA_ITEMS` lists items used for comparative analysis. The forecasting horizon (`HORIZON`) is set to 14 time steps (two weeks), while `LOOKBACK` defines the length of historical data used for training models (84 time steps).  `CONTEXT` specifies the amount of history provided as input to foundation models, and `N_ORIGINS` determines the number of starting points for a walk-forward validation scheme (12 weekly origins). Finally, `QUANTILES` defines the levels for probabilistic forecasting evaluation.

Next, the code adjusts the plotting style using Matplotlib’s `seaborn-v0_8` theme and sets a default figure size based on the dimensions defined in `CFG`. It then seeds NumPy's random number generator with the value from `CFG.SEED` to ensure consistent results across runs.

The script determines the computational device (`DEVICE`) to be used: if an Apple GPU with Metal Performance Shaders (MPS) is available, it will utilize that; otherwise, it defaults to the CPU. A message indicating which device is being used is printed to the console.

# Utils

In [ ]:
def forecast_metrics(actual, predicted):
    """Point-forecast accuracy: MAE and RMSE, rounded for display."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    return {'mae': round(float(mae), 2), 'rmse': round(float(rmse), 2)}

This code defines a function called `forecast_metrics` that calculates and returns common metrics for evaluating the accuracy of time series forecasts.

The function takes two arguments: `actual`, representing the true values, and `predicted`, representing the forecasted values. It first converts both inputs to NumPy arrays with a floating-point data type to ensure accurate calculations. 

It then computes the Mean Absolute Error (MAE) by taking the average of the absolute differences between the actual and predicted values.  The Root Mean Squared Error (RMSE) is calculated by squaring the differences, averaging them, and then taking the square root.

Finally, the function returns a dictionary containing the MAE and RMSE values, rounded to two decimal places for easier readability. This provides a concise summary of forecast accuracy.

In [ ]:
def pinball_loss(actual, quantile_forecast, tau):
    """Score a single quantile forecast at level tau (lower is better)."""
    actual = np.asarray(actual, dtype=float)
    quantile_forecast = np.asarray(quantile_forecast, dtype=float)
    diff = actual - quantile_forecast
    return float(np.mean(np.maximum(tau * diff, (tau - 1) * diff)))


def interval_coverage(actual, lower, upper):
    """Fraction of actual values that fall inside [lower, upper]."""
    actual = np.asarray(actual, dtype=float)
    inside = (actual >= np.asarray(lower, dtype=float)) & \
             (actual <= np.asarray(upper, dtype=float))
    return float(np.mean(inside))

This code defines two functions for evaluating probabilistic forecasts – specifically, quantile forecasts and prediction intervals.

The first function, `pinball_loss`, calculates the pinball loss (also known as quantile loss) for a single quantile forecast at a given level (`tau`). This is a common scoring rule used in quantile regression to assess how well a predicted quantile matches the actual value. It converts both the actual and forecasted values into NumPy arrays of floating-point numbers. The function then computes the difference between the actual and predicted values, and returns the average of the maximum of `tau` times the difference and `(tau - 1)` times the difference. A lower pinball loss indicates a better forecast.

The second function, `interval_coverage`, determines the fraction of actual values that fall within a specified prediction interval defined by a lower bound (`lower`) and an upper bound (`upper`). It converts the actual, lower, and upper bounds to NumPy arrays with floating-point data types. The code then creates a boolean array indicating whether each actual value falls between the lower and upper bounds. Finally, it returns the mean of this boolean array, which represents the proportion of actual values covered by the interval. A higher coverage indicates better calibration of the prediction intervals.

In [ ]:
def make_windows(series_z, lookback, horizon):
    """Slide a (lookback -> horizon) window over a 1-D array."""
    X, Y = [], []
    for i in range(len(series_z) - lookback - horizon + 1):
        X.append(series_z[i:i + lookback])
        Y.append(series_z[i + lookback:i + lookback + horizon])
    return np.array(X), np.array(Y)

This code defines a function called `make_windows` that prepares time series data for supervised learning by creating input-output pairs using a sliding window approach.

The function takes three arguments: `series_z`, which is the one-dimensional time series array; `lookback`, representing the length of the historical data used as input (the number of past time steps); and `horizon`, indicating the length of the future period to be predicted. 

It initializes two empty lists, `X` and `Y`. The code then iterates through the time series using a loop that stops early enough to allow for both the lookback window and the horizon to fit within the data. In each iteration, it extracts a segment of length `lookback` from the input series starting at index `i` and appends it to the `X` list (this becomes the input feature). It also extracts a subsequent segment of length `horizon` starting immediately after the lookback window and appends it to the `Y` list (this is the target variable, what we want to predict).

Finally, the function converts both `X` and `Y` lists into NumPy arrays and returns them. The resulting `X` array contains a sequence of input windows, and the `Y` array contains the corresponding future values that each window aims to predict.

In [ ]:
def context_for(item, origin, n_days):
    """The last n_days of an item's history strictly BEFORE origin."""
    history = panel[item].loc[:origin - pd.Timedelta(days=1)]
    return history.to_numpy()[-n_days:]


def actuals_for(item, origin):
    """The HORIZON true values of an item, starting AT origin."""
    window = panel[item].loc[origin:origin + pd.Timedelta(days=CFG.HORIZON - 1)]
    return window.to_numpy()

These two functions are designed to extract specific segments of time series data from a larger dataset, referred to as `panel`, based on an item ID and a given origin date. They’re used for preparing input features and target variables for forecasting.

The function `context_for` retrieves the historical context for a particular `item` up to a specified `origin` date. It accesses the time series data for that item from the `panel`. Crucially, it selects all data *up to but not including* the origin date using `.loc[:origin - pd.Timedelta(days=1)]`.  It then extracts the last `n_days` of this historical data as a NumPy array and returns it. This represents the input history used for forecasting.

The function `actuals_for` retrieves the actual observed values for an `item` over the forecast horizon, starting *at* the given `origin` date. It accesses the time series data for that item from the `panel`.  It selects a window of length equal to `CFG.HORIZON` (defined earlier as 14 days) starting at the origin date using `.loc[origin:origin + pd.Timedelta(days=CFG.HORIZON - 1)]`. It then converts this window into a NumPy array and returns it, representing the true values that will be compared against the forecasts.

In [ ]:
def plot_forecast(item, origin, forecast, title,
                  lower=None, upper=None, history_days=90):
    """History tail + actuals + forecast, with an optional 80% band."""
    history = panel[item].loc[:origin - pd.Timedelta(days=1)].tail(history_days)
    test_index = pd.date_range(origin, periods=CFG.HORIZON, freq='D')
    actual = panel[item].loc[test_index]

    plt.figure()
    plt.plot(history.index, history.values, linewidth=2, label='History')
    plt.plot(test_index, actual.values, linewidth=2,
             color='black', label='Actual')
    plt.plot(test_index, forecast, 'g--', marker='o', label='Forecast')
    if lower is not None:
        plt.fill_between(test_index, lower, upper, color='coral',
                         alpha=0.3, label='80% interval')
    plt.title(title)
    plt.legend()
    plt.xlabel("")
    plt.show()

This code defines a function called `plot_forecast` that generates a visualization of time series forecasts alongside historical data and actual values.

The function takes several arguments: `item` (the ID of the item being forecasted), `origin` (the starting date for the forecast period), `forecast` (the predicted values), `title` (the plot title), optional `lower` and `upper` bounds for a prediction interval, and `history_days` (the number of historical data points to display).

It first retrieves the recent history of the item from the `panel`, selecting data up to but not including the origin date and then taking only the last `history_days`. It also extracts the actual observed values during the forecast horizon, starting at the origin date. 

The code then creates a Matplotlib figure and plots three lines: the historical data (in blue), the actual values (in black), and the forecasted values (in green with markers). If lower and upper bounds for a prediction interval are provided, it fills the area between them with a semi-transparent coral color. 

Finally, it sets the plot title, adds a legend, removes the x-axis label, and displays the plot using `plt.show()`. This provides a clear visual comparison of the forecast against the historical context and actual outcomes.

# Groundwork

In [ ]:
raw = pd.read_csv(CFG.data_folder + 'train.csv', parse_dates=['date'])
store = raw[raw['store'] == CFG.STORE]
panel = store.pivot(index='date', columns='item', values='sales').asfreq('D')

print("Panel shape:", panel.shape)
print("Date range:", panel.index.min().date(), "to", panel.index.max().date())
print("Missing days:", int(panel.isna().sum().sum()))
panel.head(3)

This code loads and preprocesses time series data from a CSV file into a panel-like structure suitable for forecasting.

It begins by reading the `train.csv` file located in the directory specified by `CFG.data_folder` using `pd.read_csv`. The `parse_dates=['date']` argument ensures that the 'date' column is correctly interpreted as datetime objects. 

Next, it filters the data to include only records for the store identified by `CFG.STORE`.  The code then pivots this filtered data, transforming it into a panel where dates are the index, items are columns, and sales values fill the cells. The `.asfreq('D')` part ensures that there is one entry per day, even if some days had no recorded sales in the original dataset; missing values will be filled with `NaN`.

The code then prints several summary statistics about the resulting panel: its shape (number of dates and items), the range of dates covered by the data, and the total number of missing daily sales records. Finally, it displays the first three rows of the panel using `.head(3)` to provide a glimpse of the data structure. This allows for quick verification that the data has been loaded and transformed correctly.

In [ ]:
hero = panel[CFG.HERO_ITEM]
hero.plot(linewidth=2, xlabel="",
          title=f"Store {CFG.STORE}, item {CFG.HERO_ITEM}: daily sales, 2013-2017")
print(f"Mean daily sales: {hero.mean():.2f} | min: {hero.min()} | max: {hero.max()}")

This code focuses on a single item – the “hero” item – and visualizes its time series data along with some descriptive statistics.

It first selects the time series for the hero item from the `panel` using `CFG.HERO_ITEM` as the column index. Then, it creates a line plot of this time series using Matplotlib’s `.plot()` method. The plot is configured to have a linewidth of 2, no x-axis label, and a title that includes the store number (`CFG.STORE`) and item number (`CFG.HERO_ITEM`), along with a description of the data (daily sales from 2013-2017).

Finally, it calculates and prints descriptive statistics for the hero item’s time series: the mean daily sales (formatted to two decimal places), the minimum sales value, and the maximum sales value. This provides a quick overview of the typical sales range and central tendency for this particular item.

In [ ]:
hero.tail(120).plot(linewidth=2, xlabel="",
                    title="The last 120 days: the weekly rhythm up close")
plt.show()

This code displays a zoomed-in view of the hero item’s time series data, focusing on the most recent 120 days to highlight any short-term patterns or seasonality.

It selects the last 120 data points from the `hero` time series using `.tail(120)`. It then creates a line plot of these values with a linewidth of 2 and no x-axis label. The title of the plot is set to “The last 120 days: the weekly rhythm up close”, suggesting that the purpose is to observe any recurring weekly patterns in the data.

Finally, `plt.show()` displays the generated plot. This allows for a detailed examination of recent sales trends and potential seasonality within the hero item’s time series.

In [ ]:
SPLIT_DATE = pd.Timestamp('2017-10-01')
origins = pd.date_range(SPLIT_DATE, periods=CFG.N_ORIGINS, freq='7D')
train_panel = panel[panel.index < SPLIT_DATE]

print("Training days per series:", len(train_panel))
print("First origin:", origins[0].date(), "| last origin:", origins[-1].date())
print("Last forecast day:", (origins[-1] + pd.Timedelta(days=CFG.HORIZON - 1)).date())

This code sets up the training and validation data split for time series forecasting using a walk-forward validation approach.

It defines `SPLIT_DATE` as October 1, 2017, which serves as the boundary between the training and testing periods. It then generates a sequence of dates called `origins` representing the starting points for each forecast origin. These origins are spaced seven days apart (`freq='7D'`) and there are `CFG.N_ORIGINS` (defined earlier as 12) such origins, beginning at `SPLIT_DATE`.

The code then creates a training panel (`train_panel`) by selecting all data from the original `panel` where the date is strictly before `SPLIT_DATE`. This ensures that the training data does not include any information about the future. 

Finally, it prints several summary statistics: the number of days in the training dataset for each time series, the first and last dates used as forecast origins, and the date of the final forecast day (calculated by adding `CFG.HORIZON - 1` days to the last origin). This provides a clear overview of the data split and validation setup.

In [ ]:
origin0 = origins[0]
actual0 = actuals_for(CFG.HERO_ITEM, origin0)
context0 = context_for(CFG.HERO_ITEM, origin0, CFG.CONTEXT)

# repeat the last observed value for 14 days
naive_forecast = np.repeat(context0[-1], CFG.HORIZON)
# repeat the last observed week, twice
snaive_forecast = np.tile(context0[-7:], 3)[:CFG.HORIZON]

print("Naive:         ", forecast_metrics(actual0, naive_forecast))
print("Seasonal naive:", forecast_metrics(actual0, snaive_forecast))

This code evaluates the performance of two simple baseline forecasting methods – a naive forecast and a seasonal naive forecast – on the first validation origin.

It begins by selecting the first origin date (`origin0`) from the `origins` array. It then retrieves the actual observed values for the hero item during the forecast horizon starting at this origin using the `actuals_for` function, storing them in `actual0`.  The historical context for the hero item up to the origin is also retrieved using the `context_for` function and stored in `context0`.

Next, it generates two baseline forecasts. The `naive_forecast` simply repeats the last observed value from the historical context (`context0[-1]`) for each of the 14 days in the forecast horizon.  The `snaive_forecast` repeats the last seven observed values (representing one week) twice and then truncates the result to the length of the forecast horizon (`CFG.HORIZON`).

Finally, it evaluates the accuracy of both forecasts using the `forecast_metrics` function, comparing them against the actual values (`actual0`). The Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) are printed for each method, providing a benchmark for more sophisticated forecasting models.

In [ ]:
plot_forecast(CFG.HERO_ITEM, origin0, snaive_forecast,
              "Seasonal naive: repeat the last week")

This code generates a visualization of the seasonal naive forecast for the hero item at the first validation origin.

It calls the `plot_forecast` function with the following arguments: `CFG.HERO_ITEM` (the ID of the hero item), `origin0` (the starting date for the forecast period), `snaive_forecast` (the seasonal naive forecast generated earlier), and a title indicating that the plot shows the results of repeating the last week’s data.

The `plot_forecast` function will then create a plot displaying the historical data, actual values during the forecast horizon, and the seasonal naive forecast, allowing for a visual assessment of its performance compared to the true values.

# The specialist paradigm: TYOM


## A linear forecaster

In [ ]:
train_stats = {item: (float(train_panel[item].mean()), float(train_panel[item].std()))
               for item in CFG.ARENA_ITEMS}

X_list, Y_list = [], []
for item in CFG.ARENA_ITEMS:
    mean, std = train_stats[item]
    z = (train_panel[item].to_numpy() - mean) / std
    X_item, Y_item = make_windows(z, CFG.LOOKBACK, CFG.HORIZON)
    X_list.append(X_item)
    Y_list.append(Y_item)

X_all = np.concatenate(X_list).astype('float32')
Y_all = np.concatenate(Y_list).astype('float32')

# hold out the most recent 10% of each item's windows for validation
offsets = np.cumsum([0] + [len(x) for x in X_list[:-1]])
val_idx = np.concatenate([np.arange(len(x))[-int(0.1 * len(x)):] + off
                          for x, off in zip(X_list, offsets)])
val_mask = np.zeros(len(X_all), dtype=bool)
val_mask[val_idx] = True
X_train, Y_train = X_all[~val_mask], Y_all[~val_mask]
X_val, Y_val = X_all[val_mask], Y_all[val_mask]

print("Training windows:", X_train.shape, "| validation windows:", X_val.shape)

This code prepares the training and validation data for a supervised learning model by standardizing the time series data and creating sliding windows. It focuses on the items specified in `CFG.ARENA_ITEMS`.

It first calculates the mean and standard deviation of each item’s historical sales data from the `train_panel` and stores them in the `train_stats` dictionary. This is done to standardize the time series, making them more comparable and potentially improving model performance.

Then, it iterates through each item in `CFG.ARENA_ITEMS`. For each item, it standardizes the sales data by subtracting the mean and dividing by the standard deviation. It then uses the `make_windows` function to create input-output pairs (X and Y) using a sliding window approach with lengths defined by `CFG.LOOKBACK` and `CFG.HORIZON`. The resulting X and Y arrays for each item are appended to the `X_list` and `Y_list`, respectively.

Next, it concatenates all the individual X arrays in `X_list` into a single array `X_all` and similarly concatenates all the Y arrays into `Y_all`. The data type of both arrays is explicitly set to ‘float32’.

To create a validation set, the code identifies the last 10% of windows for each item as the validation data. It calculates offsets to correctly index the concatenated X_list and creates a boolean mask (`val_mask`) indicating which samples belong to the validation set. Finally, it splits `X_all` and `Y_all` into training sets (`X_train`, `Y_train`) and validation sets (`X_val`, `Y_val`) based on this mask.

Finally, it prints the shapes of the training and validation datasets, providing a summary of the data split. This ensures that there is sufficient data for both training and evaluating the model.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
with keras.device(DEVICE):
    linear_model = keras.Sequential([
        keras.layers.Input((CFG.LOOKBACK,)),
        keras.layers.Dense(CFG.HORIZON)
    ])
linear_model.compile(optimizer='adam', loss='mse')

start = time.time()
early_stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
history_linear = linear_model.fit(X_train, Y_train,
                                  validation_data=(X_val, Y_val),
                                  epochs=50, batch_size=256,
                                  callbacks=[early_stop], verbose=0)
linear_train_time = time.time() - start

print(f"Parameters: {linear_model.count_params():,}")
print(f"Training: {linear_train_time:.1f}s"
      f" ({len(history_linear.history['loss'])} epochs)")

This code trains a simple linear model using Keras for time series forecasting.

It first sets the random seed for Keras to `CFG.SEED` to ensure reproducibility of results. It then uses a context manager (`with keras.device(DEVICE):`) to specify that the model should be created and trained on the device determined earlier (either 'mps' or 'cpu').

A sequential Keras model named `linear_model` is defined. This model consists of an input layer with a shape matching the length of the lookback window (`CFG.LOOKBACK`), followed by a single dense (fully connected) layer that outputs predictions for the forecast horizon (`CFG.HORIZON`).

The model is compiled using the 'adam' optimizer and Mean Squared Error ('mse') as the loss function. 

The code then measures the training time using `time.time()`. An early stopping callback is configured to stop training if the validation loss does not improve for 5 epochs, restoring the best weights encountered during training. The model is trained using the `fit` method with the training data (`X_train`, `Y_train`), validation data (`X_val`, `Y_val`), a maximum of 50 epochs, and a batch size of 256. Verbose output is suppressed by setting `verbose=0`.

After training, the total training time is calculated and printed along with the number of epochs actually trained (obtained from the history object). The number of trainable parameters in the model is also printed for informational purposes. This provides a summary of the training process and model complexity.

In [ ]:
def forecast_all_origins(model, needs_3d=False):
    """Forecast every (origin, item) pair with a trained window model."""
    contexts, keys = [], []
    for origin in origins:
        for item in CFG.ARENA_ITEMS:
            mean, std = train_stats[item]
            z_context = (context_for(item, origin, CFG.LOOKBACK) - mean) / std
            contexts.append(z_context)
            keys.append((origin, item))

    batch = np.array(contexts, dtype='float32')
    if needs_3d:
        batch = batch[..., None]
    predictions = model.predict(batch, verbose=0)

    forecasts = {}
    for key, pred in zip(keys, predictions):
        mean, std = train_stats[key[1]]
        forecasts[key] = pred * std + mean
    return forecasts

This code defines a function called `forecast_all_origins` that generates forecasts for all combinations of validation origins and items using a trained Keras model.

The function takes the trained model (`model`) as input, along with an optional boolean argument `needs_3d` indicating whether the model expects 3D input data (which is not used in this specific case).

It initializes two empty lists: `contexts` to store the standardized historical context for each forecast and `keys` to store tuples representing the origin date and item ID. It then iterates through all validation origins in the `origins` array and all items in `CFG.ARENA_ITEMS`. For each (origin, item) pair, it retrieves the historical context using the `context_for` function, standardizes it using the pre-calculated mean and standard deviation from `train_stats`, and appends it to the `contexts` list along with a tuple representing the origin and item to the `keys` list.

The standardized contexts are then converted into a NumPy array (`batch`) of type ‘float32’. If `needs_3d` is True, an extra dimension is added to the batch (although this isn’t used in the current implementation). The model's `predict` method is called with this batch to generate forecasts.

Finally, it creates a dictionary called `forecasts` to store the de-standardized predictions. It iterates through the keys and corresponding predictions, de-standardizing each prediction by multiplying it by the standard deviation and adding the mean for that item (obtained from `train_stats`). The resulting de-standardized forecasts are stored in the `forecasts` dictionary with the origin-item tuple as the key. This function returns the `forecasts` dictionary containing all generated predictions.

In [ ]:
linear_forecasts = forecast_all_origins(linear_model)

hero_key = (origin0, CFG.HERO_ITEM)
print("Linear on the close-up series:",
      forecast_metrics(actual0, linear_forecasts[hero_key]))
plot_forecast(CFG.HERO_ITEM, origin0, linear_forecasts[hero_key],
              "Linear specialist (trained from scratch)")

This code generates forecasts for all validation origins and items using the trained linear model and then evaluates its performance on the hero item.

It calls the `forecast_all_origins` function with the trained `linear_model` to generate predictions for every combination of origin date and item in `CFG.ARENA_ITEMS`. The resulting forecasts are stored in the `linear_forecasts` dictionary.

Next, it retrieves the key corresponding to the hero item and the first validation origin (`origin0`) from this dictionary. It then evaluates the accuracy of the linear model’s forecast for the hero item using the `forecast_metrics` function, comparing the predicted values with the actual observed values (`actual0`). The MAE and RMSE are printed to the console.

Finally, it generates a visualization of the linear model's forecast for the hero item at the first validation origin using the `plot_forecast` function. The plot is titled "Linear specialist (trained from scratch)" to indicate that this forecast was generated by the simple linear model trained from scratch. This allows for visual assessment of the model’s performance on a specific time series.

## A GRU forecaster

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
with keras.device(DEVICE):
    gru_model = keras.Sequential([
        keras.layers.Input((CFG.LOOKBACK, 1)),
        keras.layers.GRU(64),
        keras.layers.Dense(CFG.HORIZON)
    ])
gru_model.compile(optimizer='adam', loss='mse')

start = time.time()
early_stop = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
history_gru = gru_model.fit(X_train[..., None], Y_train,
                            validation_data=(X_val[..., None], Y_val),
                            epochs=30, batch_size=256,
                            callbacks=[early_stop], verbose=0)
gru_train_time = time.time() - start

print(f"Parameters: {gru_model.count_params():,}")
print(f"Training: {gru_train_time:.1f}s"
      f" ({len(history_gru.history['loss'])} epochs)")

This code trains a GRU (Gated Recurrent Unit) neural network using Keras for time series forecasting.

It begins by setting the random seed for Keras to `CFG.SEED` to ensure reproducibility. It then uses a context manager (`with keras.device(DEVICE):`) to specify that the model should be created and trained on the selected device (either 'mps' or 'cpu').

A sequential Keras model named `gru_model` is defined. This model consists of an input layer with a shape matching the lookback window length (`CFG.LOOKBACK`) and one additional dimension for a single feature, followed by a GRU layer with 64 units, and finally a dense (fully connected) layer that outputs predictions for the forecast horizon (`CFG.HORIZON`).

The model is compiled using the 'adam' optimizer and Mean Squared Error ('mse') as the loss function.

The code then measures the training time using `time.time()`. An early stopping callback is configured to stop training if the validation loss does not improve for 3 epochs, restoring the best weights encountered during training. The model is trained using the `fit` method with the training data (`X_train[..., None]`, `Y_train`), validation data (`X_val[..., None]`, `Y_val`), a maximum of 30 epochs, and a batch size of 256.  The `[..., None]` part adds an extra dimension to the input data, which is required by the GRU layer because it expects 3D tensors (samples, time steps, features). Verbose output is suppressed by setting `verbose=0`.

After training, the total training time is calculated and printed along with the number of epochs actually trained. The number of trainable parameters in the model is also printed for informational purposes. This provides a summary of the training process and model complexity.

In [ ]:
gru_forecasts = forecast_all_origins(gru_model, needs_3d=True)

print("GRU on the close-up series:",
      forecast_metrics(actual0, gru_forecasts[hero_key]))
plot_forecast(CFG.HERO_ITEM, origin0, gru_forecasts[hero_key],
              "GRU specialist (trained from scratch)")

This code generates forecasts for all validation origins and items using the trained GRU model and then evaluates its performance on the hero item.

It calls the `forecast_all_origins` function with the trained `gru_model`, passing `needs_3d=True` because the GRU model expects 3D input data. The resulting forecasts are stored in the `gru_forecasts` dictionary.

Next, it retrieves the key corresponding to the hero item and the first validation origin (`origin0`) from this dictionary. It then evaluates the accuracy of the GRU model’s forecast for the hero item using the `forecast_metrics` function, comparing the predicted values with the actual observed values (`actual0`). The MAE and RMSE are printed to the console.

Finally, it generates a visualization of the GRU model's forecast for the hero item at the first validation origin using the `plot_forecast` function. The plot is titled "GRU specialist (trained from scratch)" to indicate that this forecast was generated by the GRU neural network trained from scratch. This allows for visual assessment of the model’s performance on a specific time series and comparison with the linear model's results.

# How a foundation model reads a series

SFSG 

Before running the foundation models, it pays to understand the one problem every
single one of them had to solve first. A transformer is a machine for sequences of
**tokens** — discrete symbols from a finite vocabulary, each mapped to an embedding
vector. Language arrives pre-tokenized; a time series does not. It is a stream of
continuous floats with no vocabulary, arbitrary scale (sales of 20 a day here, server
requests in the millions elsewhere), and a noise level that makes any single
observation nearly meaningless. Feeding one float per token is also computationally
hostile: self-attention costs grow with the *square* of sequence length.

The field converged on two main answers, and the models we are about to run split
neatly along this line — so we build both answers ourselves, in a few lines of NumPy,
on our own series. (A third answer, constructing tokens from *lagged values* at
seasonal offsets — the route taken by **Lag-Llama**, an early open model from
ServiceNow/Mila — preserves exact values and encodes periodicity explicitly, but has
lost ground to the two below and we won't dwell on it.)

## Patching: sub-sequences as tokens

In [ ]:
PATCH_LEN = 32
patches = context0.reshape(-1, PATCH_LEN)

print(f"{len(context0)} days -> {patches.shape[0]} patches (tokens) "
      f"of {PATCH_LEN} days each")
print(f"Attention cost vs. one-token-per-day: "
      f"{(patches.shape[0] ** 2) / (len(context0) ** 2):.4%}")

plt.figure()
for i in range(6):
    segment = patches[-6 + i]
    plt.plot(np.arange(i * PATCH_LEN, (i + 1) * PATCH_LEN), segment,
             linewidth=2, label=f"patch {patches.shape[0] - 6 + i + 1}")
plt.title("The last six patches: each 32-day block becomes one token")
plt.xlabel("day within the plotted stretch")
plt.legend()
plt.show()

**Patching** — the answer chosen by TimesFM and most of the field. One `reshape` cuts
the 512-day context into **16 non-overlapping blocks of
32 days**; inside the real model, each block is pushed through a
small MLP to become one embedding vector, and *those* are the tokens the transformer
attends over. The printout shows the economics: 16 tokens instead of 512 shrinks the
attention matrix to **under 0.1%** of its one-token-per-day size — this is what makes
16k-point contexts feasible at all.

The plot shows the semantic argument, which matters as much as the economics. Each
coloured segment — one patch — is a month of data containing four full weekly cycles
and a stretch of the seasonal slope. Like a word compared to a letter, a patch is a
unit *worth attending to*: a single day's value is mostly noise, but a 32-day shape is
a recognisable pattern. The transformer then reasons about relationships between
month-shaped things, not day-shaped things.

## Quantization: sales as sentences

In [ ]:
N_BINS = 4096
scale = np.mean(np.abs(context0))
scaled = context0 / scale

bin_edges = np.linspace(scaled.min(), scaled.max(), N_BINS + 1)
tokens = np.clip(np.digitize(scaled, bin_edges) - 1, 0, N_BINS - 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
reconstructed = bin_centers[tokens] * scale

round_trip_rmse = float(np.sqrt(np.mean((context0 - reconstructed) ** 2)))
print("Vocabulary size:", N_BINS)
print("Tokens actually used by this series:", len(np.unique(tokens)))
print("First ten tokens:", tokens[:10])
print("Round-trip error (RMSE, sales units):", round(round_trip_rmse, 4))

**Quantization** — the opposite philosophy, introduced by the original Amazon
**Chronos**: if transformers are language machines, turn the series *into language*.
Three steps, each one line here. **Mean scaling** divides the series by its mean
absolute value (23.17 for our context), collapsing sales-of-20 and
requests-in-millions onto a common scale. **Binning** chops the scaled range into
`N_BINS = 4096` intervals — the vocabulary — and `np.digitize` assigns each day its
bin id (the `- 1` and `np.clip` convert digitize's 1-based edge convention into safe
0-based token ids). The series is now literally a sentence of integers, and the
printout shows the first ten "words". A T5 language model can be trained on such
sentences with an ordinary cross-entropy loss, predicting a *distribution over the
vocabulary* for the next token — which is where Chronos' probabilistic forecasts
originally came from.

The cost of discretisation turns out to be trivial: decoding each token back to its
bin centre reproduces the series with an RMSE of
**0.002** sales units — invisible next to day-to-day noise. Note
also how few words this series needs: only **37 of the
4,096 tokens** are ever used. The vocabulary is sized for the diversity of a whole
pre-training universe, not one series. The real weakness is subtler than precision
loss: token-by-token generation is slow, and values outside the binned range simply
cannot be represented — one reason the Chronos family later moved on, as we will see.

In [ ]:
plt.figure()
plt.step(np.arange(120), tokens[-120:], linewidth=2)
plt.title("The last 120 days as a language model would read them")
plt.xlabel("day within context")
plt.ylabel("token id")
plt.show()

The same 120 days we plotted in the groundwork section, now as the token sequence a
language model would consume. The shape is perfectly recognisable — weekly seesaw,
autumn downslope — just written in a 4,096-symbol alphabet instead of floats. This
picture is the entire Chronos hypothesis in one image: *if the structure survives
tokenization, a language model can learn its grammar.*

# The zero-shot ladder

Concepts in hand, we climb through three foundation models, in order of increasing
architectural distance from what we have already built by hand this season. All three
are open-weights, download from Hugging Face, and run on this laptop's GPU. For each
we follow an identical protocol: load, forecast **all 120 (origin, item) tasks** with a
512-day context — timing the whole thing — then inspect the close-up series. Nobody
gets fine-tuned; nobody sees a single training gradient from our data.

| Model | Maker | Backbone | Params | Tokenization | Probabilistic output |
|---|---|---|---|---|---|
| **TimesFM 2.5** | Google | decoder-only transformer | 200M | patches (float) | continuous quantile head |
| **TiRex** | NX-AI | stacked xLSTM (no attention) | 35M | patches + masking | 9-level quantile grid |
| **Chronos-2** | Amazon | transformer w/ group attention | 120M | patches (float) | direct multi-quantile |

Three families from the wider zoo are worth knowing about even though we won't run
them. **Moirai 2.0** (Salesforce) pursues "universal forecasting" hardest: its
any-variate attention flattens an arbitrary number of variables into one sequence, and
its Mixture-of-Experts variant matched dense rivals with up to 65× fewer *active*
parameters; tellingly, version 2.0 abandoned its original masked-encoder design for a
decoder-only one — the whole field has converged there. **Toto 2.0** (Datadog)
specialises in high-cardinality observability telemetry, with a scaling family reaching
2.5B parameters trained on over a trillion points, and two ideas worth stealing:
*causal* patch normalisation (statistics computed only over past patches, so
normalisation itself cannot leak the future) and an arcsinh transform that tames
metrics spanning orders of magnitude. **TimeGPT** (Nixtla) is the API-first option — no
weights, just a client library — with conformal prediction intervals; a pragmatic
choice when you want zero infrastructure, at the price of sending your data out and
trusting a black box. Honourable mentions: **MOMENT** (CMU), a masked encoder built for
classification, imputation and anomaly detection as much as forecasting, and
**Time-MoE**, which scaled sparse mixture-of-experts to 2.4B parameters on 300 billion
time points to show that scaling laws hold for time series too.

## TimesFM 2.5: the patched decoder

In [ ]:
start = time.time()
timesfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch")
timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=CFG.CONTEXT,
        max_horizon=64,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        fix_quantile_crossing=True,
    )
)
print(f"Loaded and compiled in {time.time() - start:.1f}s")

**TimesFM** is Google Research's flagship: a decoder-only transformer over patch
tokens, pre-trained on roughly 100 billion real-world time points — much of it Google
Trends and Wikipedia pageview series, chosen precisely because web attention encodes
human weekly and holiday rhythms that transfer to domains like retail. Version 2.5,
the current one, is leaner and stronger than its predecessors: 200M parameters (down
from 500M in 2.0), a context window stretched to 16,384 points, and no more manual
frequency flag — earlier versions made you declare "this is daily data", 2.5 infers it.

Its architectural signature is the **asymmetric patch**: input patches of 32 points
but *output* patches of 128. A conventional autoregressive model forecasting 256 steps
needs 256 generation steps, each feeding on its own possibly-wrong output; TimesFM
emits 128-point blocks, so long horizons take a couple of forward passes instead of
hundreds — faster, and with less error compounding.

The two-stage setup mirrors its API: `from_pretrained` pulls the checkpoint, then
`compile` fixes an inference configuration. `max_context=512` matches our diet;
`normalize_inputs` lets the model handle scaling internally (the reason we can feed
raw sales while our Keras models needed hand-built z-scoring);
`use_continuous_quantile_head` activates an optional 30M-parameter head that emits
deciles alongside the point forecast — quantile forecasting as an architectural
add-on; and `fix_quantile_crossing` post-processes those deciles so the 90th never
dips below the 10th, a small but real pathology of independently-predicted quantiles.

In [ ]:
timesfm_point, timesfm_quantiles = {}, {}

start = time.time()
for origin in origins:
    inputs = [context_for(item, origin, CFG.CONTEXT).astype('float32')
              for item in CFG.ARENA_ITEMS]
    point, quantiles = timesfm_model.forecast(horizon=CFG.HORIZON, inputs=inputs)
    for j, item in enumerate(CFG.ARENA_ITEMS):
        timesfm_point[(origin, item)] = point[j]
        timesfm_quantiles[(origin, item)] = quantiles[j]  # (14, 10): mean, q10..q90
timesfm_time = time.time() - start

print(f"All {len(origins) * len(CFG.ARENA_ITEMS)} two-week forecasts "
      f"in {timesfm_time:.1f}s - zero training")

The entire "modeling" workflow of the new paradigm is this loop: hand over raw history,
receive forecasts. `forecast` takes a plain list of NumPy arrays — one per series, no
windowing, no scaling, no feature engineering — and returns two arrays per series: a
point forecast of shape (14,) and a quantile block of shape (14, 10), whose columns
are the mean followed by the ten deciles q10 through q90. We stash both in
dictionaries keyed by (origin, item), the exact shape our specialists' forecasts live
in, so the arena can score everyone identically.

The punchline is the clock: **all 120 two-week forecasts in about
6.5 seconds**, having never seen a gradient from our data. Recall
the GRU needed 23.7s of training before it could forecast anything —
and that cost recurs for every new dataset, while this one never does.

In [ ]:
q = timesfm_quantiles[hero_key]
print("TimesFM 2.5:", forecast_metrics(actual0, timesfm_point[hero_key]))
print("80% interval coverage:",
      round(interval_coverage(actual0, q[:, 1], q[:, 9]), 2))

plot_forecast(CFG.HERO_ITEM, origin0, timesfm_point[hero_key],
              "TimesFM 2.5, zero-shot", lower=q[:, 1], upper=q[:, 9])

The close-up verdict: RMSE **5.55** — comfortably better than
seasonal naive (6.53) and the GRU (6.64),
a shade behind the linear specialist (5.04), from a model that
learned everything it knows about retail from *other* time series. The plot shows a
genuinely learned forecast, not a copied week: smooth weekly seesaw, correctly
depressed level, and — new for this series — a coral uncertainty band from the
quantile head.

The band is the thing to scrutinise. Its coverage here is
**64%** against the nominal 80% — over fourteen days,
that's a couple of escapees more than promised, and too small a sample to judge; the
arena will pool 168 days per model before ruling on calibration. Note the band's
*shape* though: it widens visibly with horizon, day 14 admitting more doubt than day
1. That fan is textbook forecasting behaviour, produced zero-shot.

## TiRex: recurrence strikes back

In [ ]:
start = time.time()
tirex_model = load_tirex("NX-AI/TiRex-2", device=DEVICE)
print(f"Loaded in {time.time() - start:.1f}s")

tirex_mean, tirex_quantiles = {}, {}
start = time.time()
for origin in origins:
    contexts = np.stack([context_for(item, origin, CFG.CONTEXT)
                         for item in CFG.ARENA_ITEMS])
    quantiles, mean = tirex_model.forecast(
        context=torch.tensor(contexts, dtype=torch.float32),
        prediction_length=CFG.HORIZON)
    for j, item in enumerate(CFG.ARENA_ITEMS):
        tirex_quantiles[(origin, item)] = quantiles[j].numpy()  # (14, 9): q10..q90
        tirex_mean[(origin, item)] = mean[j].numpy()
tirex_time = time.time() - start

print(f"All 120 forecasts in {tirex_time:.1f}s")

**TiRex**, from NX-AI (the Linz lab of LSTM inventor Sepp Hochreiter), is the ladder's
contrarian rung: a foundation model with **no attention at all**. Its backbone is a
stack of **xLSTM** blocks — the 2024 revival of the LSTM with exponential gating and
better parallelism — which changes the economics rather than just the aesthetics. A
transformer must re-attend over the whole context for every new observation, at cost
quadratic in context length; a recurrent model *carries its history* in a fixed-size
hidden state and absorbs each new patch at constant cost. That is why TiRex's natural
habitat is streaming and edge deployment — its makers benchmark it on industrial PLC
hardware with a few GB of RAM — and why the whole model is only **35M parameters**,
roughly a sixth of TimesFM, yet competitive with the largest entries on zero-shot
leaderboards like GIFT-Eval.

One training trick is worth knowing because we met its ancestor in the transfer
episode: **contiguous patch masking** — long masked spans that the model must
reconstruct in parallel during pre-training. At inference the forecast horizon is just
one more masked span, decoded in a single forward pass: no token-by-token generation,
no compounding of one-step errors.

The API is a two-liner: `forecast` takes a batch of raw contexts and returns a
(batch, 14, 9) quantile tensor — the levels q10 through q90 directly, no parametric
distribution in between — plus a mean. Everything lands in our standard dictionaries.
The clock says 2.4s for all 120 forecasts, and a follow-up model,
**TiRex-2**, extends the recipe to multivariate inputs and known-future covariates
(its checkpoint is gated on Hugging Face, so we run the original here).

In [ ]:
q = tirex_quantiles[hero_key]
print("TiRex:", forecast_metrics(actual0, tirex_mean[hero_key]))
print("80% interval coverage:",
      round(interval_coverage(actual0, q[:, 0], q[:, 8]), 2))

plot_forecast(CFG.HERO_ITEM, origin0, tirex_mean[hero_key],
              "TiRex, zero-shot", lower=q[:, 0], upper=q[:, 8])

RMSE **5.75** on the close-up, with 64%
of actuals inside the 80% band — statistically indistinguishable from TimesFM's
close-up showing, from a model one-sixth the size with a completely different
computational core. (Column indices differ from TimesFM only because TiRex returns
the nine quantiles without a leading mean column: index 0 is q10, index 8 is q90.)
The deeper lesson of this rung: at today's scale of pre-training data, the *backbone*
— attention versus recurrence — matters less than the recipe of patches, masking, and
massive multi-domain exposure. The 2017 architecture war ended in a draw once
everyone got the same food.

## Chronos-2: universal forecasting

In [ ]:
start = time.time()
chronos_model = BaseChronosPipeline.from_pretrained("amazon/chronos-2",
                                                    device_map=DEVICE)
print(f"Loaded in {time.time() - start:.1f}s")

chronos_mean, chronos_quantiles = {}, {}
start = time.time()
for origin in origins:
    contexts = torch.tensor(
        np.stack([context_for(item, origin, CFG.CONTEXT)
                  for item in CFG.ARENA_ITEMS]),
        dtype=torch.float32).unsqueeze(1)   # (10 series, 1 variate, 512 days)
    q_list, mean_list = chronos_model.predict_quantiles(
        contexts, prediction_length=CFG.HORIZON,
        quantile_levels=CFG.QUANTILES)
    for j, item in enumerate(CFG.ARENA_ITEMS):
        chronos_quantiles[(origin, item)] = q_list[j][0].numpy()  # (14, 3)
        chronos_mean[(origin, item)] = mean_list[j][0].numpy()
chronos_time = time.time() - start

print(f"All 120 forecasts in {chronos_time:.1f}s")

The top rung. The Chronos family began as the purest "time series is a language" bet —
the T5-with-4,096-token-vocabulary design whose tokenizer we rebuilt above. That
original was accurate but slow (sampling token by token), so **Chronos-Bolt** swapped
in patches and direct quantile prediction for a ~250× speedup — and **Chronos-2**, the
current generation we run here (120M parameters), is a near-complete redesign that
keeps only the name: continuous patch embeddings, a single pass predicting all
quantile levels directly, and — its true novelty — **universality**. One checkpoint
handles univariate series, multivariate groups, and known-future covariates, none of
which the tokenized original could represent at all. The family's own journey away
from quantization is the field's verdict on that debate.

Mechanically, note the shape discipline: `predict_quantiles` wants
`(n_series, n_variates, history)`, so ten independent items enter as `(10, 1, 512)`
via `unsqueeze(1)`. It returns per-series quantile tensors at exactly the levels we
request — no decile bookkeeping — and a mean. The clock: **all 120 forecasts in
1.8s**, the fastest of the three, from a model that would have been
the slowest in its first incarnation.

In [ ]:
q = chronos_quantiles[hero_key]
print("Chronos-2:", forecast_metrics(actual0, chronos_mean[hero_key]))
print("80% interval coverage:",
      round(interval_coverage(actual0, q[:, 0], q[:, 2]), 2))

plot_forecast(CFG.HERO_ITEM, origin0, chronos_mean[hero_key],
              "Chronos-2, zero-shot", lower=q[:, 0], upper=q[:, 2])

RMSE **5.62** on the close-up — between its two ladder-mates,
with the now-familiar learned seesaw and fanning band (columns 0 and 2 are our
requested q10 and q90). Coverage here is 57%, the lowest of
the three on this single window; again, fourteen days decide nothing. All three
foundation models have now landed in the same close-up neighbourhood — roughly 5.5 to
5.8 RMSE, versus 5.04 for the linear specialist — which is
itself the finding: three alien architectures, trained by three companies on three
different data piles, agree with each other more than they differ from the locals.

In [ ]:
all_items = list(panel.columns)

group_mean, group_quantiles = {}, {}
start = time.time()
for origin in origins:
    contexts = torch.tensor(
        np.stack([context_for(item, origin, CFG.CONTEXT)
                  for item in all_items]),
        dtype=torch.float32).unsqueeze(0)   # (1 task, 50 variates, 512 days)
    q_list, mean_list = chronos_model.predict_quantiles(
        contexts, prediction_length=CFG.HORIZON,
        quantile_levels=CFG.QUANTILES)
    for j, item in enumerate(all_items):
        if item in CFG.ARENA_ITEMS:
            group_quantiles[(origin, item)] = q_list[0][j].numpy()
            group_mean[(origin, item)] = mean_list[0][j].numpy()
group_time = time.time() - start

print(f"Group mode, 50 items jointly: {group_time:.1f}s")
print("Close-up item, forecast alone:     ",
      forecast_metrics(actual0, chronos_mean[hero_key]))
print("Close-up item, forecast in a group:",
      forecast_metrics(actual0, group_mean[hero_key]))

Chronos-2's headline feature deserves its own experiment: **in-context learning
across series**. Reshape alone changes the semantics — the same fifty series that
entered the previous cell as fifty independent tasks `(50, 1, 512)` now enter as *one*
task with fifty variates `(1, 50, 512)`. Inside the model, a group-attention layer
lets every series' representation peek at every other's, so it can pick up cross-item
structure on the fly, at inference time, with no weight updates. This is the time
series analogue of prompting an LLM with related examples — and where an LLM's
in-context learning *emerged* from scale, here it is deliberately engineered into the
architecture.

The result is a finding, not a triumph: on the close-up item the group forecast
(RMSE **5.71**) is essentially identical to the solo one
(**5.62**). And it makes sense. These fifty items are siblings
— same store, same weekly rhythm, same summer peak — so item 1's own 512 days already
contain everything its siblings could tell you; the group's information is *redundant*,
not complementary. Context sharing pays when the neighbours know something the target
doesn't: a study on 200 low-voltage electricity feeders found Chronos-2's group mode
genuinely useful when weather covariates were missing, because neighbouring feeders
*were* the weather signal by proxy. Redundant context is at least cheap to include: the twelve group calls
produce forecasts for all fifty items in 1.0s — per series, five times cheaper than
the ten-item batches above (1.8s for a fifth of the forecasts). The arena will tell us whether the pattern holds beyond
one item.

# Head-to-head: 120 forecasts, seven forecasters

Close-ups shown, anecdotes collected — time to score everything on the same 120 tasks:
ten items, twelve weekly origins, fourteen days each. Point accuracy (RMSE) for
everyone; pinball loss and 80%-interval coverage for the models that offer quantiles;
and the compute column that the zero-shot paradigm is really about — seconds of
training for the specialists versus seconds of inference for the foundation models.

In [ ]:
snaive_forecasts = {}
for origin in origins:
    for item in CFG.ARENA_ITEMS:
        last_week = context_for(item, origin, 7)
        snaive_forecasts[(origin, item)] = np.tile(last_week, 3)[:CFG.HORIZON]

# the ground truth for all 120 tasks, concatenated once and shared by both scorers
ARENA_ACTUALS = np.concatenate([actuals_for(item, o)
                                for o in origins for item in CFG.ARENA_ITEMS])

def arena_rmse(forecasts):
    """Pooled RMSE over all (origin, item) tasks."""
    preds = np.concatenate([forecasts[(o, item)]
                            for o in origins for item in CFG.ARENA_ITEMS])
    return forecast_metrics(ARENA_ACTUALS, preds)['rmse']

def arena_prob(mean_fc, quantile_fc, lo_col, med_col, hi_col):
    """RMSE + averaged pinball + 80% coverage for a quantile model."""
    stack = lambda col: np.concatenate([quantile_fc[(o, item)][:, col]
                                        for o in origins
                                        for item in CFG.ARENA_ITEMS])
    lo, med, hi = stack(lo_col), stack(med_col), stack(hi_col)
    means = np.concatenate([mean_fc[(o, item)]
                            for o in origins for item in CFG.ARENA_ITEMS])
    pinball = np.mean([pinball_loss(ARENA_ACTUALS, lo, 0.1),
                       pinball_loss(ARENA_ACTUALS, med, 0.5),
                       pinball_loss(ARENA_ACTUALS, hi, 0.9)])
    return {'rmse': forecast_metrics(ARENA_ACTUALS, means)['rmse'],
            'pinball': round(float(pinball), 2),
            'coverage_80': round(interval_coverage(ARENA_ACTUALS, lo, hi), 2)}

Scoring machinery. The seasonal naive dictionary is built the same way as everyone
else's forecasts, so it faces the identical 120 tasks. The ground truth is concatenated
once into `ARENA_ACTUALS` — all 120 × 14 = 1,680 forecast days in one long vector — and
both scorers reuse it, pooling errors across tasks rather than averaging per-task
scores, which weights every forecast day equally. `arena_prob` adds the probabilistic
side; its `stack` lambda pulls one quantile column out of every stored
(14, n_quantiles) block. The column indices are passed in per model because, as we saw,
each library arranges its quantile axis differently — TimesFM has a leading mean
column, TiRex stores nine deciles, Chronos-2 returns exactly the three levels we
requested.

In [ ]:
results = []
results.append({'model': 'Seasonal naive (7)', 'zero_shot': '-',
                'rmse': arena_rmse(snaive_forecasts),
                'pinball': np.nan, 'coverage_80': np.nan, 'compute_s': 0.0})
results.append({'model': 'Linear (trained)', 'zero_shot': 'no',
                'rmse': arena_rmse(linear_forecasts),
                'pinball': np.nan, 'coverage_80': np.nan,
                'compute_s': round(linear_train_time, 1)})
results.append({'model': 'GRU (trained)', 'zero_shot': 'no',
                'rmse': arena_rmse(gru_forecasts),
                'pinball': np.nan, 'coverage_80': np.nan,
                'compute_s': round(gru_train_time, 1)})
results.append({'model': 'TimesFM 2.5', 'zero_shot': 'yes',
                **arena_prob(timesfm_point, timesfm_quantiles, 1, 5, 9),
                'compute_s': round(timesfm_time, 1)})
results.append({'model': 'TiRex', 'zero_shot': 'yes',
                **arena_prob(tirex_mean, tirex_quantiles, 0, 4, 8),
                'compute_s': round(tirex_time, 1)})
results.append({'model': 'Chronos-2', 'zero_shot': 'yes',
                **arena_prob(chronos_mean, chronos_quantiles, 0, 1, 2),
                'compute_s': round(chronos_time, 1)})
results.append({'model': 'Chronos-2 (group)', 'zero_shot': 'yes',
                **arena_prob(group_mean, group_quantiles, 0, 1, 2),
                'compute_s': round(group_time, 1)})

results_df = pd.DataFrame(results).set_index('model')
results_df = results_df[['zero_shot', 'rmse', 'pinball', 'coverage_80', 'compute_s']]
results_df

The table of the episode. Reading it top to bottom:

- **Seasonal naive: 11.55.** The free baseline, and the
  yardstick for everything below.
- **The trained specialists: 9.46 (linear) and
  9.37 (GRU).** Both clear the baseline by a wide margin —
  and, with 120 forecasts instead of one, the close-up verdict flips: the GRU edges
  out the linear model after all. Single-window rankings deceive; walk-forward
  aggregates decide.
- **The foundation models: 8.3, 8.76,
  7.74.** All three beat both specialists, zero-shot.
  **Chronos-2 wins the arena outright** at 7.74 — roughly 17%
  better than the GRU and 33% better than seasonal naive — with TimesFM second and
  TiRex, at a sixth of the parameters, third yet still ahead of everything trained
  here. The group variant lands at 8.01: the close-up
  pattern holds, redundant sibling context does not add accuracy on this panel.
- **Calibration, the quiet star.** Coverage of the 80% bands:
  79%, 79%, and
  80% across 1,680 forecast days. All three are within a
  point of nominal — uncertainty estimates you could hand to an inventory planner
  as-is, produced for series the models had never seen. Getting intervals this honest
  from a hand-built model requires the entire conformal-prediction apparatus of
  episode seven. Pinball losses rank the same way (1.86 /
  1.98 / 2.1), so Chronos-2's win is not
  a point-forecast artefact.
- **The compute column** is the paradigm in one glance: the specialists *spent* their
  seconds before they could forecast at all and must spend them again for the next
  dataset; the foundation models' seconds are one-off inference, amortising a
  pre-training bill someone else already paid.

One caveat before declaring a new world order: fifty well-behaved daily retail series
with strong weekly seasonality is *home turf* for models pre-trained on exactly such
patterns. Hold that thought for two sections.

In [ ]:
plot_df = results_df.sort_values('rmse', ascending=False)
palette = {'yes': 'seagreen', 'no': 'steelblue', '-': 'firebrick'}
colors = [palette[z] for z in plot_df['zero_shot']]

plt.figure()
plt.barh(plot_df.index, plot_df['rmse'], color=colors)
plt.xlabel("RMSE across 120 two-week forecasts (lower is better)")
plt.title("Zero-shot models (green) vs trained specialists (blue) vs the free baseline (red)")
plt.show()

The same verdict as a picture, with the field sorted so the best forecaster ends up on
top and the colors following the series convention: red for the free baseline, blue for
specialists trained on this very dataset, green for zero-shot foundation models. Every
green bar is shorter than every blue bar, and the red one towers over both camps. The
spread *within* the greens — Chronos-2 to TiRex — is smaller than the gap between the
greens and the blues, which is the practical headline: on this kind of data, *which*
foundation model you pick matters less than *whether* you pick one.

# Application: the cold-start problem

The arena gave every model five years of history — the setting where "train your own"
is strongest. The advertised sweet spot of foundation models is the opposite corner:
a series too *young* to train on. A product launched two months ago, a sensor
installed last quarter, a store opened in spring. Simulation: we hand every approach
only the **last 60 days** of the close-up item's history before the first origin, and
ask for the same 14-day forecast. For the specialists we shrink the window to 28 days
— with 60 days you cannot even fill an 84-day lookback once.

In [ ]:
COLD_DAYS = 60
COLD_LOOKBACK = 28
cold_context = context_for(CFG.HERO_ITEM, origin0, COLD_DAYS)

cold_scores = {}
cold_scores['Seasonal naive (7)'] = forecast_metrics(
    actual0, np.tile(cold_context[-7:], 3)[:CFG.HORIZON])['rmse']

# scratch models on 60 days: standardise, window, train
cold_mean, cold_std = float(cold_context.mean()), float(cold_context.std())
z_cold = (cold_context - cold_mean) / cold_std
X_cold, Y_cold = make_windows(z_cold, COLD_LOOKBACK, CFG.HORIZON)
print("Training windows available:", X_cold.shape[0])

keras.utils.set_random_seed(CFG.SEED)
with keras.device(DEVICE):
    linear_cold = keras.Sequential([
        keras.layers.Input((COLD_LOOKBACK,)),
        keras.layers.Dense(CFG.HORIZON)
    ])
linear_cold.compile(optimizer='adam', loss='mse')
linear_cold.fit(X_cold, Y_cold, epochs=200, batch_size=8, verbose=0)
pred = linear_cold.predict(z_cold[-COLD_LOOKBACK:][None, :],
                           verbose=0)[0] * cold_std + cold_mean
cold_scores['Linear (trained on 60d)'] = forecast_metrics(actual0, pred)['rmse']

keras.utils.set_random_seed(CFG.SEED)
with keras.device(DEVICE):
    gru_cold = keras.Sequential([
        keras.layers.Input((COLD_LOOKBACK, 1)),
        keras.layers.GRU(64),
        keras.layers.Dense(CFG.HORIZON)
    ])
gru_cold.compile(optimizer='adam', loss='mse')
gru_cold.fit(X_cold[..., None], Y_cold, epochs=200, batch_size=8, verbose=0)
pred = gru_cold.predict(z_cold[-COLD_LOOKBACK:][None, :, None],
                        verbose=0)[0] * cold_std + cold_mean
cold_scores['GRU (trained on 60d)'] = forecast_metrics(actual0, pred)['rmse']

The starved specialists. Sixty days of history minus a 28-day window minus a 14-day
target leaves **19 training windows** — where the arena versions had
sixteen thousand. We train both scratch architectures anyway, giving them every
chance: 200 epochs, tiny batches, the same seeded, deviced construction ritual as
before (each gets a fresh seed call so the run is reproducible). Note the scaling
statistics now come from the 60 cold days themselves — there is no other history to
compute them from, which is itself part of the cold-start handicap: even the *mean*
is poorly estimated. This is the regime where "just fit a neural network" stops being
advice and starts being a prank.

In [ ]:
# foundation models get the same 60 days as context
point, quantiles = timesfm_model.forecast(horizon=CFG.HORIZON,
                                          inputs=[cold_context.astype('float32')])
cold_scores['TimesFM 2.5 (60d ctx)'] = forecast_metrics(actual0, point[0])['rmse']

tirex_q_cold, tirex_m_cold = tirex_model.forecast(
    context=torch.tensor(cold_context, dtype=torch.float32).unsqueeze(0),
    prediction_length=CFG.HORIZON)
cold_scores['TiRex (60d ctx)'] = forecast_metrics(
    actual0, tirex_m_cold[0].numpy())['rmse']

q_list, mean_list = chronos_model.predict_quantiles(
    torch.tensor(cold_context, dtype=torch.float32).reshape(1, 1, -1),
    prediction_length=CFG.HORIZON, quantile_levels=CFG.QUANTILES)
cold_scores['Chronos-2 (60d ctx)'] = forecast_metrics(
    actual0, mean_list[0][0].numpy())['rmse']

# references from the full-history world, same origin and item
cold_scores['GRU (trained, full history)'] = forecast_metrics(
    actual0, gru_forecasts[hero_key])['rmse']
cold_scores['Chronos-2 (512d ctx)'] = forecast_metrics(
    actual0, chronos_mean[hero_key])['rmse']

cold_df = pd.DataFrame.from_dict(cold_scores, orient='index',
                                 columns=['rmse']).sort_values('rmse')
cold_df

The foundation models take the identical 60-day array — the calls are unchanged except
for the shorter context — and two full-history references join the table for
perspective. The ranking tells the cold-start story cleanly:

- **The zero-shot models degrade gracefully.** TiRex (5.91) and
  Chronos-2 (6.11) lead the 60-day field — and TiRex on 60 days
  even beats the GRU trained on *five years* (6.64).
  Sixty days is two months of weekly cycles: enough for a model that has seen a
  million weekly rhythms to lock on.
- **The starved specialists don't.** The 60-day linear model
  (8.47) finishes *behind seasonal naive*
  (6.53) — nineteen windows cannot estimate an honest 28×14
  map — and the 60-day GRU (6.74) essentially ties the free
  baseline. Training on a cold series buys you complexity risk, not accuracy.
- **History still helps the history-lovers.** Chronos-2 with its full 512-day context
  (5.62) remains the best number in the table — a foundation
  model is not a reason to throw history away; it is a way to survive not having any.

Standard caveat: this is one origin of one item — a demonstration of the mechanism,
not a benchmark. (The mechanism replicates: the pattern above is exactly what the
literature reports at scale.)

In [ ]:
q_cold = tirex_q_cold[0].numpy()
plot_forecast(CFG.HERO_ITEM, origin0, tirex_m_cold[0].numpy(),
              "TiRex, zero-shot from only 60 days of history",
              lower=q_cold[:, 0], upper=q_cold[:, 8],
              history_days=COLD_DAYS)

The cold-start close-up, with everything the model was given on screen: sixty days in,
a calibrated two-week forecast out — weekly seesaw in place, level tracked, honest
uncertainty band around it. Rewind to the start of this series and consider what this
plot would have taken in episode one: identify the seasonality, difference the trend,
pick orders, fit, diagnose, iterate. Whatever reservations the next section raises,
*this* — competent forecasts on data-starved series, for free — is the genuinely new
capability foundation models brought to the table.

# The evaluation crisis

Time to spoil the party, carefully. Our arena result — three zero-shot models beating
honest local specialists — mirrors the headline claims of every foundation-model
paper. But the fine print of those claims has become its own research topic, and three
issues deserve a place in any practitioner's head.

**Data leakage.** Zero-shot means *these weights never saw this series* — but who
checks? The pre-training corpora are scraped from the same public archives (Monash,
GluonTS, Kaggle, M-competitions) that everyone benchmarks on. Our own arena is not
above suspicion: the store-item dataset is a popular Kaggle competition from 2018,
public for years before any of these models were trained. None of the three publishes
a manifest that would let us rule out its presence in their corpora. When a foundation
model aces a famous public benchmark, some of that performance may be memory, not
generalisation. The field's responses: benchmarks with enforced cutoffs (GIFT-Eval,
fev-bench), evaluation on data published *after* a model's training cutoff (the
electricity-feeder study we cited did exactly this), and — most directly —
decontaminated checkpoints like TiRex-2's `-g` and `-f` variants, retrained with the
benchmark datasets explicitly excluded. Your own private data remains the only
benchmark you can fully trust.

**The baseline problem.** A recurring, embarrassing literature finding: on stable,
low-frequency series — monthly sales with clean seasonality, say — statistical
workhorses (ETS, Theta, seasonal naive) still match or beat billion-parameter models,
at a millionth of the cost. Our arena, with its 11.55-RMSE
naive baseline soundly beaten, is favourable terrain for the big models: daily
frequency, long history, subtle trend interactions. Move to twenty-four points of
monthly data and the picture can invert. No leaderboard exempts you from running the
free baseline on *your* data first.

**Benchmarks are not deployments.** Pooled RMSE on 1,680 forecast days is a fine
scorecard and a poor business metric. The feeder study made this point sharply by
scoring peak-load errors against the physical damage curve of a grid fuse — the
forecast errors that *cost money* are not the ones RMSE emphasises. Before promoting
any model, foundation or local, translate its errors into the units your decisions
are made in.

# Where this leaves us

The season-long arc of this series has been a march of increasing machinery — and this
episode's twist is that the machinery has left the building. What we established
today, on real data and a level playing field:

1. **Zero-shot forecasting works.** Three foundation models with nothing but a 512-day
   context beat specialists trained on the very panel being forecast:
   7.74 / 8.3 / 8.76 RMSE
   versus 9.37 for the GRU and
   11.55 for seasonal naive — with near-nominal 80%
   coverage thrown in for free.
2. **The tokenization question has an answer.** Patching won; the quantization
   pioneer's own successor abandoned the vocabulary. The backbone question, by
   contrast, ended in a draw — a 35M-parameter recurrent model hangs with
   200M-parameter transformers, so pick by deployment shape, not ideology.
3. **Cold start is the killer app.** Sixty days of history was enough for zero-shot
   models to beat a fully-trained GRU; it was nowhere near enough to train anything.
4. **Skepticism is part of the toolkit.** Leakage, friendly benchmarks, and RMSE
   myopia all inflate foundation-model claims; private data and free baselines deflate
   them.

A practical routing rule to leave with, distilled from the deployment literature:
**central business forecasting** over many series with decent history — a
patch-transformer like TimesFM 2.5 or Chronos-2, with covariates when you have them;
**streaming and edge**, where memory and latency are the constraint — a recurrent
model like TiRex; **high-cardinality machine telemetry** — the observability
specialists like Toto 2.0; **no infrastructure at all** — a managed API like TimeGPT,
if your data may leave the house. And in every one of these cells: seasonal naive
first, always, because the day a fifty-year-old heuristic beats your foundation model,
you want to be the one who finds out.

Fine-tuning these models on your own panel — the middle ground between zero-shot and
from-scratch that we explored for ordinary networks in the transfer-learning episode —
is where we point the telescope next.